<a href="https://colab.research.google.com/github/Chetalam/cnn-skin-cancer-classification/blob/main/notebooks/08_scc_decision_and_fitzpatrick_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 — SCC decision experiment and Fitzpatrick audit
Run all cells in a fresh Colab runtime. No training or test inference is performed. This notebook tests a **predeclared small SCC decision-weight grid on validation only**, and audits recorded Fitzpatrick types. It does not generate new patients, infer skin type from image brightness, or establish clinical readiness. Experiment 6 and its original test results remain unchanged.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
CLASSES = ['BCC', 'Melanoma', 'SCC']
RUN = 'exp06_7fe97493900f'
SOURCE = ROOT / 'Reports' / 'Training' / RUN / 'ResNet50'
PACKAGE = ROOT / 'Models' / 'Selected' / ('final_review_' + RUN)
OUT = ROOT / 'Reports' / 'SCC_and_Fitzpatrick' / RUN
OUT.mkdir(parents=True, exist_ok=True)
meta = json.loads((PACKAGE / 'metadata.json').read_text())
assert meta['run_id'] == RUN and meta['class_names'] == CLASSES
assert hashlib.sha256((PACKAGE / 'model.keras').read_bytes()).hexdigest() == meta['model_sha256']
frames = {}
for split in ['train', 'validation', 'test']:
    path = ROOT / 'Datasets' / 'Splits' / f'{split}_metadata.csv'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == meta['manifest_hashes'][split], 'Split changed'
    frames[split] = pd.read_csv(path, dtype={'image_id':str, 'patient_id':str})
    assert frames[split]['class_label'].map(dict(zip(CLASSES, range(3)))).eq(frames[split]['label_index']).all()
print('Verified existing model and unchanged split manifests.')


Mounted at /content/drive
Verified existing model and unchanged split manifests.


## SCC decision experiment
A multiplier increases or decreases the SCC score before choosing a class. Adjusted scores are **decision scores, not calibrated probabilities**. Select by validation macro F1, retaining the unadjusted rule on ties. Do not choose a multiplier using test results. The existing test set has already informed development; independent evaluation is still required.


In [2]:
def class_metrics(labels, predicted):
    rows = []
    for i, name in enumerate(CLASSES):
        tp = int(((labels == i) & (predicted == i)).sum())
        fp = int(((labels != i) & (predicted == i)).sum())
        fn = int(((labels == i) & (predicted != i)).sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
        rows.append(dict(class_label=name, support=int((labels == i).sum()),
                         precision=precision, recall=recall, f1=f1, tp=tp, fp=fp, fn=fn))
    return pd.DataFrame(rows)

# Align saved validation probabilities using explicit dataset/image identifiers.
predictions = pd.read_csv(SOURCE / 'validation_predictions.csv', dtype={'image_id':str})
print('Saved validation columns:', predictions.columns.tolist())
keys = ['dataset', 'image_id']
if not set(keys).issubset(predictions.columns):
    raise ValueError('Saved predictions need dataset and image_id; never assume row order.')
if predictions.duplicated(keys).any():
    raise ValueError('Duplicate saved prediction identifiers.')
score_columns = ['score_' + name for name in CLASSES]
if not set(score_columns).issubset(predictions.columns):
    # Training exports may use probability_<class> instead of score_<class>.
    alternatives = ['probability_' + name for name in CLASSES]
    if not set(alternatives).issubset(predictions.columns):
        raise ValueError('Expected score_<class> or probability_<class> columns. Printed columns identify the export format.')
    predictions = predictions.rename(columns=dict(zip(alternatives, score_columns)))
val = frames['validation'].merge(predictions[keys + score_columns], on=keys, how='left', validate='one_to_one')
if val[score_columns].isna().any().any():
    raise ValueError('Some validation identifiers lack scores.')
scores = val[score_columns].to_numpy(float)
assert np.isfinite(scores).all() and (scores >= 0).all()
np.testing.assert_allclose(scores.sum(axis=1), 1, atol=1e-5)
y = val['label_index'].to_numpy(int)
GRID = [1.0, 0.5, 0.75, 1.25, 1.5, 2.0, 3.0]
rows = []
for weight in GRID:
    decision_scores = scores * np.array([1., 1., weight])
    predicted = decision_scores.argmax(axis=1)
    table = class_metrics(y, predicted)
    scc = table.iloc[2]
    rows.append(dict(scc_multiplier=weight, validation_accuracy=float((y == predicted).mean()),
                     validation_macro_f1=float(table.f1.mean()), SCC_precision=scc.precision,
                     SCC_recall=scc.recall, SCC_f1=scc.f1))
comparison = pd.DataFrame(rows)
best = comparison.sort_values('validation_macro_f1', ascending=False, kind='stable').iloc[0]
comparison.to_csv(OUT / 'validation_decision_comparison.csv', index=False)
display(comparison.round(4))
rule = dict(experiment='SCC validation decision rule', source_run_id=RUN,
            model_sha256=meta['model_sha256'], manifest_hashes=meta['manifest_hashes'],
            class_names=CLASSES, scc_multiplier=float(best.scc_multiplier),
            selection_basis='validation macro F1 only; baseline wins exact ties',
            grid=GRID, validation_macro_f1=float(best.validation_macro_f1),
            test_evaluated=False, calibrated_probabilities=False)
rule_path = OUT / 'decision_rule.json'
if rule_path.exists() and json.loads(rule_path.read_text()) != rule:
    raise ValueError('A different frozen decision rule already exists; preserve it in a separate experiment folder.')
rule_path.write_text(json.dumps(rule, indent=2))
val['adjusted_predicted_class'] = np.asarray(CLASSES)[(scores * [1, 1, rule['scc_multiplier']]).argmax(axis=1)]
val.to_csv(OUT / 'validation_decisions.csv', index=False)
class_metrics(y, (scores * [1, 1, rule['scc_multiplier']]).argmax(axis=1)).to_csv(OUT / 'validation_adjusted_per_class.csv', index=False)
val.loc[(val.class_label == 'SCC') | (val.adjusted_predicted_class == 'SCC')].to_csv(OUT / 'scc_validation_review.csv', index=False)
print('Frozen validation-selected SCC multiplier:', rule['scc_multiplier'])
print('No test predictions were used to select this rule. Existing model package unchanged.')


Saved validation columns: ['id', 'dataset', 'image_id', 'lesion_id', 'patient_id', 'original_label', 'class_label', 'age', 'sex', 'localization', 'image_path', 'image_hash', 'fitzpatrick', 'fitzpatrick_source', 'biopsy_confirmed', 'diagnosis_method', 'group_id', 'label_index', 'split', 'predicted_class', 'score_BCC', 'score_Melanoma', 'score_SCC']


,scc_multiplier,validation_accuracy,validation_macro_f1,SCC_precision,SCC_recall,SCC_f1
0,1.00,0.8769,0.7056,0.4000,0.2308,0.2927
1,0.50,0.8821,0.6711,0.4286,0.1154,0.1818
2,0.75,0.8769,0.6658,0.3333,0.1154,0.1714
3,1.25,0.8744,0.7009,0.3529,0.2308,0.2791
4,1.50,0.8641,0.6892,0.2857,0.2308,0.2553
5,2.00,0.8513,0.6854,0.2500,0.2692,0.2593
6,3.00,0.8256,0.6895,0.2391,0.4231,0.3056


Frozen validation-selected SCC multiplier: 1.0
No test predictions were used to select this rule. Existing model package unchanged.


## Recorded Fitzpatrick audit
Join the original PAD metadata by filename, preserving unknown values for HAM10000 and missing PAD metadata. Report I–III and IV–VI as descriptive groupings, plus individual recorded types. These are recorded phototypes; they do not establish ethnicity or independently verified skin tone. Scores below describe existing records only. Small or absent class support must remain visible.


In [3]:
pad = pd.read_csv(ROOT / 'Datasets' / 'PAD-UFES-20' / 'metadata.csv', dtype={'img_id':str})
pad['image_id'] = pad['img_id'].str.strip()
if pad['image_id'].duplicated().any():
    raise ValueError('PAD metadata contains duplicate image filenames.')
phototype = pd.to_numeric(pad['fitspatrick'], errors='coerce')
if (phototype.notna() & (~phototype.isin(range(1,7)))).any():
    raise ValueError('Unexpected recorded Fitzpatrick values.')
pad['recorded_fitzpatrick'] = phototype.astype('Int64').astype('string').fillna('Unknown')
pad['dataset'] = 'PAD-UFES-20'
lookup = pad[['dataset','image_id','recorded_fitzpatrick']]
def attach_phototype(frame):
    frame = frame.drop(columns=['recorded_fitzpatrick'], errors='ignore').copy()
    frame['image_id'] = frame['image_id'].astype(str).str.strip()
    result = frame.merge(lookup, on=['dataset','image_id'], how='left', validate='many_to_one')
    result['recorded_fitzpatrick'] = result['recorded_fitzpatrick'].fillna('Unknown')
    result['phototype_group'] = result['recorded_fitzpatrick'].map({
        '1':'I-III','2':'I-III','3':'I-III','4':'IV-VI','5':'IV-VI','6':'IV-VI'
    }).fillna('Unknown')
    return result
all_records = attach_phototype(pd.concat(frames.values(), ignore_index=True))
coverage = all_records.groupby(['split','dataset','class_label','recorded_fitzpatrick'], dropna=False).size().rename('image_count').reset_index()
coverage.to_csv(OUT / 'fitzpatrick_coverage.csv', index=False)
display(coverage)
display(pd.crosstab([all_records['split'],all_records['class_label']], all_records['phototype_group']))

def subgroup_metrics(frame, predicted_column, split, rule_name):
    records=[]
    for grouping in ['recorded_fitzpatrick', 'phototype_group']:
        for value, subset in frame.groupby(grouping, dropna=False):
            labels=subset.class_label.map(dict(zip(CLASSES,range(3)))).to_numpy(int)
            predicted=subset[predicted_column].map(dict(zip(CLASSES,range(3))))
            if predicted.isna().any(): raise ValueError('Unknown predicted class')
            table=class_metrics(labels,predicted.to_numpy(int))
            for record in table.to_dict('records'):
                # Absent true classes have undefined recall/F1 for subgroup interpretation.
                if record['support']==0:
                    record['recall']=None
                    record['f1']=None
                records.append(dict(split=split, decision_rule=rule_name, grouping=grouping,
                                    subgroup=str(value), subgroup_images=len(subset),
                                    patient_count=subset.patient_id.nunique() if 'patient_id' in subset else None,
                                    insufficient_support=record['support'] < 20, **record))
    return records
base_val=attach_phototype(val)
base_val['baseline_predicted_class']=np.asarray(CLASSES)[scores.argmax(axis=1)]
subgroups=subgroup_metrics(base_val,'baseline_predicted_class','validation','original_argmax')
subgroups+=subgroup_metrics(base_val,'adjusted_predicted_class','validation','validation_selected_multiplier')
# Read existing original test outputs for descriptive subgroup reporting only.
# Never use these results to choose a multiplier or modify diagnoses.
test_path=ROOT / 'Reports' / 'Evaluation' / PACKAGE.name / 'test_predictions.csv'
if test_path.exists():
    test_pred=pd.read_csv(test_path,dtype={'image_id':str})
    if test_pred.duplicated(['dataset','image_id']).any(): raise ValueError('Duplicate test predictions')
    joined=frames['test'].merge(test_pred[['dataset','image_id','predicted_class']],on=['dataset','image_id'],how='left',validate='one_to_one')
    if joined.predicted_class.isna().any(): raise ValueError('Missing original test predictions')
    subgroups+=subgroup_metrics(attach_phototype(joined),'predicted_class','test','original_argmax_already_evaluated')
subgroup_table=pd.DataFrame(subgroups)
subgroup_table.to_csv(OUT / 'fitzpatrick_subgroup_metrics.csv',index=False)
display(subgroup_table.round(4))
summary = """# SCC and Fitzpatrick audit

The decision multiplier was selected using validation only. No new test evaluation or retraining was performed.

Fitzpatrick results describe recorded metadata; missing phototypes remain Unknown. Subgroups with fewer than 20 class examples are flagged for descriptive caution; 20 is a reporting flag, not a clinical validation standard. Zero-support class recall and F1 remain undefined. No skin-type fairness claim is established.

The four hmnist CSVs are not additional independent patient data. No synthetic images or skin-color transformations were added. Additional representative real SCC cases and independent evaluation remain required.
"""
(OUT / 'audit_summary.md').write_text(summary)
print('SCC decision experiment and Fitzpatrick audit code completed.')
print('Reports saved to:',OUT)
print('Integration requirements are NOT declared met by this notebook.')


,split,dataset,class_label,recorded_fitzpatrick,image_count
0,test,HAM10000,BCC,Unknown,72
1,test,HAM10000,Melanoma,Unknown,173
2,test,PAD-UFES-20,BCC,1,10
3,test,PAD-UFES-20,BCC,2,74
4,test,PAD-UFES-20,BCC,3,40
5,test,PAD-UFES-20,BCC,4,1
6,test,PAD-UFES-20,Melanoma,2,6
7,test,PAD-UFES-20,Melanoma,3,1
8,test,PAD-UFES-20,Melanoma,4,2
9,test,PAD-UFES-20,SCC,2,20


phototype_group         I-III  IV-VI  Unknown
split      class_label                       
test       BCC            124      1       72
           Melanoma         7      2      173
           SCC             28      1        0
train      BCC            569     21      366
           Melanoma        34      2      783
           SCC            127      3        0
validation BCC            121      3       76
           Melanoma         7      0      157
           SCC             25      1        0

,split,decision_rule,grouping,subgroup,subgroup_images,patient_count,insufficient_support,class_label,support,precision,recall,f1,tp,fp,fn
0,validation,original_argmax,recorded_fitzpatrick,1,11,10,True,BCC,8,0.8000,1.0000,0.8889,8,2,0
1,validation,original_argmax,recorded_fitzpatrick,1,11,10,True,Melanoma,0,0.0000,NaN,NaN,0,0,0
2,validation,original_argmax,recorded_fitzpatrick,1,11,10,True,SCC,3,1.0000,0.3333,0.5000,1,0,2
3,validation,original_argmax,recorded_fitzpatrick,2,95,63,False,BCC,76,0.8395,0.8947,0.8662,68,13,8
4,validation,original_argmax,recorded_fitzpatrick,2,95,63,True,Melanoma,4,1.0000,1.0000,1.0000,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,test,original_argmax_already_evaluated,phototype_group,IV-VI,4,3,True,Melanoma,2,1.0000,1.0000,1.0000,2,0,0
77,test,original_argmax_already_evaluated,phototype_group,IV-VI,4,3,True,SCC,1,0.0000,0.0000,0.0000,0,0,1
78,test,original_argmax_already_evaluated,phototype_group,Unknown,245,0,False,BCC,72,0.8667,0.9028,0.8844,65,10,7
79,test,original_argmax_already_evaluated,phototype_group,Unknown,245,0,False,Melanoma,173,0.9588,0.9422,0.9504,163,7,10


SCC decision experiment and Fitzpatrick audit code completed.
Reports saved to: /content/drive/MyDrive/cnn-skin-cancer-classification/Reports/SCC_and_Fitzpatrick/exp06_7fe97493900f
Integration requirements are NOT declared met by this notebook.


## Save
Drive reports are written automatically. Save this executed notebook to Drive and to `notebooks/08_scc_decision_and_fitzpatrick_audit.ipynb` on branch `codex/final-training-review`. Commit message: `Record SCC decision experiment and Fitzpatrick subgroup audit`. The original model and original reports remain intact. Report actual findings even if the decision adjustment does not improve SCC.
